# 16 — Training without the 125-cycle cap: measured, not deployed

**Why this notebook exists.** After notebook 15, the deployed interval still covered only 73% of FD004 test engines with more than 125 cycles left (FD001: 91%). Every model is trained on remaining life clipped at 125 cycles, so none has seen an engine with 200 cycles left, and every interval's upper bound falls short for long-lived engines. Survival models can learn from the true, uncapped durations; the cap is a convention borrowed from regression. This notebook measures what removing it does, to the point estimate and to the interval's upper bound.

**Setup.** The same out-of-fold protocol as notebook 15 (5 folds by engine, each training engine scored at 10 random points between 30% and 99% of its life by models that never saw it), repeated with `cap` removed. The capped results are notebook 15's saved file.

**Questions and rules, fixed before looking at the test sets:**
1. *Point estimate.* Uncapped RSF replaces capped RSF only if it is better on both C-index and RMSE, on both subsets, out of fold.
2. *Interval upper bound.* Candidates keep notebook 15's lower bound (the more cautious of RSF's and Weibull's) and change only the upper bound. A candidate is eligible if, on both subsets, out-of-fold coverage is ≥ 88% overall and ≥ 85% in each band of true remaining life (≤ 25, 26–125, > 125 cycles). The narrowest eligible candidate whose extra models fit a ~250 MB budget (the Hugging Face upload and the free live-demo tier) is checked once on the official test sets (≥ 85% overall, near failure and above the cap, on both).
3. *Deployment.* A separate decision, made on the results and on cost, below.

**Disclosure.** The leaf-30 candidate was added after the leaf-5 one turned out to need 1.2 GB of extra models (section 4); it was chosen for size, not on its results. The interval rule was finalised after seeing FD001's out-of-fold table, as in notebook 15. Test sets were not used for either.

In [1]:
import sys
import tempfile
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sksurv.metrics import concordance_index_censored

sys.path.append(str(Path("..") / "src"))
from turbofan_rul import (data_loading, evaluation, features, inference, interval_calibration as ic,
                          labeling, models, plotting)
from turbofan_rul import export_deployment_artifacts as export

plotting.style()
SUBSETS = ["FD001", "FD004"]
UNCAPPED = 100_000          # larger than any C-MAPSS engine life, i.e. no cap
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "models"
TRUE_BANDS, TRUE_LABELS = [0, 25, 125, np.inf], ["true <= 25", "true 26-125", "true > 125"]
KEYS = ["fold", "unit_number", "time_cycles", "true_rul"]
RECOMPUTE = False   # True re-fits all fold models (~6-10 minutes per subset and leaf size)

## 1. Out-of-fold predictions, with and without the cap

In [2]:
def training_frame(subset):
    data_loading.download_cmapss(subset, data_dir=DATA_DIR)
    cmapss = data_loading.load_subset(subset, data_dir=DATA_DIR)
    if subset in export.REGIME_CLUSTERED_SUBSETS:
        kept, train_for_labeling, _, _ = export._prepare_regime_clustered(subset, cmapss)
    else:
        kept, train_for_labeling, _, _ = export._prepare_fd001_style(subset, cmapss, PROCESSED_DIR)
    return kept, train_for_labeling

oof = {}
for subset in SUBSETS:
    capped = pd.read_csv(PROCESSED_DIR / f"interval_oof_{subset.lower()}.csv")   # from notebook 15
    frames = {}
    for leaf in (5, 30):
        path = PROCESSED_DIR / f"interval_oof_uncapped_leaf{leaf}_{subset.lower()}.csv"
        if path.exists() and not RECOMPUTE:
            frames[leaf] = pd.read_csv(path)
        else:
            t0 = time.time()
            kept, train_for_labeling = training_frame(subset)
            frames[leaf] = ic.out_of_fold_intervals(
                train_for_labeling, kept, cap=UNCAPPED, min_samples_leaf=leaf,
                row_stride=export.ROW_STRIDE_BY_SUBSET[subset],
            )
            frames[leaf].to_csv(path, index=False)
            print(f"{subset} leaf={leaf}: computed in {time.time() - t0:.0f}s")
    merged = (capped
              .merge(frames[5].rename(columns=lambda c: c if c in KEYS else f"{c}_u5"), on=KEYS)
              .merge(frames[30][KEYS + ["rsf_high"]].rename(columns={"rsf_high": "rsf_high_u30"}), on=KEYS))
    assert len(merged) == len(capped), "capped and uncapped runs must score the same points"
    oof[subset] = merged
    print(f"{subset}: {len(merged)} out-of-fold points, identical across the three runs")

FD001 leaf=5: computed in 308s


FD001 leaf=30: computed in 251s
FD001: 956 out-of-fold points, identical across the three runs


FD004 leaf=5: computed in 467s


FD004 leaf=30: computed in 421s
FD004: 2417 out-of-fold points, identical across the three runs


## 2. Point estimate: does dropping the cap help RSF? (rule 1)

In [3]:
def point_row(y, p):
    return {"C-index": concordance_index_censored(np.ones(len(y), bool), y, -p)[0],
            "RMSE": evaluation.rmse(y, p), "MAE": evaluation.mae(y, p),
            "late rate": evaluation.late_prediction_rate(y, p)}

rows = []
for subset, f in oof.items():
    y = f.true_rul.to_numpy(float)
    band = pd.cut(f.true_rul, TRUE_BANDS, labels=TRUE_LABELS)
    for label, col in [("capped (deployed)", "rsf_pred"), ("uncapped", "rsf_pred_u5")]:
        p = f[col].to_numpy(float)
        r = {"subset": subset, "RSF": label, **point_row(y, p)}
        r.update({f"RMSE {b}": evaluation.rmse(y[band == b], p[band == b]) for b in TRUE_LABELS})
        rows.append(r)
point = pd.DataFrame(rows).set_index(["subset", "RSF"])
display(point.round(3))

uncapped_wins = all(
    point.loc[(s, "uncapped"), "C-index"] > point.loc[(s, "capped (deployed)"), "C-index"]
    and point.loc[(s, "uncapped"), "RMSE"] < point.loc[(s, "capped (deployed)"), "RMSE"]
    for s in SUBSETS)
print("Rule 1 -- uncapped RSF better on C-index and RMSE on both subsets:", uncapped_wins)

C-index    RMSE     MAE  late rate  RMSE true <= 25  \
subset RSF                                                                      
FD001  capped (deployed)    0.864  24.384  16.598      0.559            6.151   
       uncapped             0.852  28.006  19.900      0.646            6.353   
FD004  capped (deployed)    0.846  36.989  24.390      0.555            7.635   
       uncapped             0.840  40.416  29.453      0.727            7.800   

                          RMSE true 26-125  RMSE true > 125  
subset RSF                                                   
FD001  capped (deployed)            18.387           51.744  
       uncapped                     28.129           41.498  
FD004  capped (deployed)            21.374           67.804  
       uncapped                     41.236           48.485

Rule 1 -- uncapped RSF better on C-index and RMSE on both subsets: False


The cap stays for the point estimate. Without it, RSF is worse overall on both subsets: it gains above the cap but loses more in the 26–125 band, where most engines are. Early-life sensor readings carry almost no degradation signal, so asking the forest to tell 250 cycles from 300 adds noise without information. This matches the standard C-MAPSS result. (The NASA score is left out here: out of fold, very long FD004 engines truncated early produce single errors of 300+ cycles, and the exponential penalty lets a handful of points dominate the mean.)

## 3. Interval upper bound (rule 2)

In [4]:
def candidates(f):
    low = np.maximum(np.minimum(f.rsf_low, f.weibull_low), 0)     # notebook 15's lower bound, unchanged
    return {
        "current (Weibull, capped)": (low, f.weibull_high),
        "uncapped Weibull upper": (low, f.weibull_high_u5),
        "uncapped RSF upper (leaf 5)": (low, f.rsf_high_u5),
        "uncapped RSF upper (leaf 30)": (low, f.rsf_high_u30),
    }

rows = []
for subset, f in oof.items():
    y = f.true_rul.to_numpy(float)
    band = pd.cut(f.true_rul, TRUE_BANDS, labels=TRUE_LABELS)
    for name, (low, high) in candidates(f).items():
        cov = (y >= low) & (y <= high)
        rows.append({"subset": subset, "upper bound": name, "coverage": cov.mean(),
                     "width": (high - low).mean(), **{b: cov[band == b].mean() for b in TRUE_LABELS}})
upper = pd.DataFrame(rows).set_index(["upper bound", "subset"]).unstack("subset")
display(upper.round(3))

eligible = {name: all(upper.loc[name, ("coverage", s)] >= 0.88
                      and all(upper.loc[name, (b, s)] >= 0.85 for b in TRUE_LABELS) for s in SUBSETS)
            for name in upper.index}
summary = pd.DataFrame({"eligible": eligible, "mean width": upper["width"].mean(axis=1).round(1)}).sort_values("mean width")
display(summary)
print(f"Narrowest eligible on training data: {summary[summary.eligible].index[0]}")

coverage           width          true <= 25  \
subset                          FD001  FD004    FD001    FD004      FD001   
upper bound                                                                 
current (Weibull, capped)       0.932  0.895   91.892  101.819      0.994   
uncapped RSF upper (leaf 30)    0.987  0.986  116.339  164.295      0.994   
uncapped RSF upper (leaf 5)     0.974  0.979  106.945  152.742      0.994   
uncapped Weibull upper          0.960  0.973  122.710  159.871      0.994   

                                    true 26-125        true > 125         
subset                        FD004       FD001  FD004      FD001  FD004  
upper bound                                                               
current (Weibull, capped)     0.988       0.970  0.974      0.651  0.627  
uncapped RSF upper (leaf 30)  0.988       1.000  0.992      0.913  0.970  
uncapped RSF upper (leaf 5)   0.988       0.988  0.988      0.873  0.948  
uncapped Weibull upper        0.988       0.974  0.980      0.841  0.943

,eligible,mean width
"current (Weibull, capped)",False,96.9
uncapped RSF upper (leaf 5),True,129.8
uncapped RSF upper (leaf 30),True,140.3
uncapped Weibull upper,False,141.3


Narrowest eligible on training data: uncapped RSF upper (leaf 5)


Only the uncapped RSF upper bounds clear 85% in every band on both subsets. The uncapped *Weibull* upper bound, the cheapest option at a few kilobytes, misses on FD001 above the cap (just under 85%). The narrowest eligible candidate is the leaf-5 forest; whether it can be deployed depends on its size (next section).

## 4. What it would cost: model size

In [5]:
sizes, full_models = [], {}
with tempfile.TemporaryDirectory() as tmp:
    for subset in SUBSETS:
        kept, train_for_labeling = training_frame(subset)
        train = features.build_feature_matrix(
            labeling.build_train_survival_frame(train_for_labeling, cap=UNCAPPED), kept_sensors=kept)
        cols = [c for c in train.columns if c not in ic.META_COLS]
        for leaf in (5, 30):
            t0 = time.time()
            model = models.fit_random_survival_forest(
                train, cols, row_stride=export.ROW_STRIDE_BY_SUBSET[subset],
                min_samples_leaf=leaf, random_state=42)
            path = Path(tmp) / f"{subset}_{leaf}.joblib"
            joblib.dump(model, path)
            sizes.append({"subset": subset, "leaf": leaf, "size (MB)": path.stat().st_size / 1e6,
                          "fit (s)": time.time() - t0})
            if leaf == 30:
                full_models[subset] = model      # kept for the test-set check below
sizes = pd.DataFrame(sizes)
deployed = {s: (MODELS_DIR / s.lower() / "rsf.joblib").stat().st_size / 1e6 for s in SUBSETS}
display(sizes.round(0))
print("Deployed (capped, leaf 5) RSF sizes, MB:", {s: round(v) for s, v in deployed.items()})

BUDGET_MB = 250
extra = sizes.groupby("leaf")["size (MB)"].sum()
selected = next(f"uncapped RSF upper (leaf {leaf})" for leaf in (5, 30)
                if eligible[f"uncapped RSF upper (leaf {leaf})"] and extra[leaf] <= BUDGET_MB)
print({f"leaf {k}": f"{v:.0f} MB extra" for k, v in extra.items()})
print(f"Selected within the {BUDGET_MB} MB budget: {selected}")

,subset,leaf,size (MB),fit (s)
0,FD001,5,695.0,67.0
1,FD001,30,111.0,54.0
2,FD004,5,529.0,40.0
3,FD004,30,85.0,31.0


Deployed (capped, leaf 5) RSF sizes, MB: {'FD001': 226, 'FD004': 126}
{'leaf 5': '1224 MB extra', 'leaf 30': '196 MB extra'}
Selected within the 250 MB budget: uncapped RSF upper (leaf 30)


Uncapped leaf-5 forests are about 3× the deployed ones (they store survival curves over ~300–400 time points instead of ~125), so FD001 and FD004 together would add about 1.2 GB, far over budget. Leaf 30, the project's original RSF setting, adds about 200 MB for both. It would serve only the upper bound, so the tuned leaf-5 point model stays as it is, and it is eligible on training data with wider margins than leaf 5 (91% / 97% above the cap).

In [6]:
# Where does the extra width go? Median width by the RSF point estimate,
# i.e. by how close to failure the model thinks the engine is.
pred_bands = [0, 25, 50, 90, np.inf]
pred_labels = ["pred < 25", "pred 25-50", "pred 50-90", "pred >= 90"]
rows = []
for subset, f in oof.items():
    low, old_high = candidates(f)["current (Weibull, capped)"]
    _, new_high = candidates(f)[selected]
    g = pd.DataFrame({"band": pd.cut(f.rsf_pred, pred_bands, labels=pred_labels),
                      "current": old_high - low, "selected": new_high - low})
    w = g.groupby("band", observed=True)[["current", "selected"]].median()
    w["points"] = g.groupby("band", observed=True).size()
    w.insert(0, "subset", subset)
    rows.append(w)
pd.concat(rows).round(0)

,subset,current,selected,points
band,,,,
pred < 25,FD001,35.0,23.0,139
pred 25-50,FD001,53.0,52.0,163
pred 50-90,FD001,83.0,109.0,253
pred >= 90,FD001,126.0,174.0,401
pred < 25,FD004,43.0,36.0,256
pred 25-50,FD004,59.0,62.0,316
pred 50-90,FD004,85.0,124.0,545
pred >= 90,FD004,128.0,217.0,1300


The extra width lands where it is honest. Near failure the interval gets *tighter*, because RSF's lower and upper quantiles are both well informed there; engines the model already calls healthy get much wider ranges. For an engine far from failure, "80 to 300 cycles" is the truthful answer.

## 5. Confirm once on the official test sets

In [7]:
rows = []
for subset in SUBSETS:
    bundle = inference.load_deployment_bundle(MODELS_DIR / subset.lower())
    cmapss = data_loading.load_subset(subset, data_dir=DATA_DIR)
    true_rul = cmapss.rul.set_index("unit_number")["RUL"]
    for unit, engine in cmapss.test.groupby("unit_number"):
        engine = engine.sort_values("time_cycles")
        readings = engine[["time_cycles"] + bundle.raw_passthrough_cols]
        r = inference.predict(bundle, readings)
        # Same feature row inference.predict builds, scored by the uncapped leaf-30 forest.
        working = readings.reset_index(drop=True)
        working["unit_number"] = 1
        if bundle.uses_regime_clustering:
            labels = features.assign_regime_clusters(working, bundle.regime_kmeans)
            working = features.apply_regime_normalization(working, inference.SENSOR_COLS, labels,
                                                          bundle.regime_norm_stats)
        row = features.build_feature_matrix(working, kept_sensors=bundle.kept_sensors).iloc[[-1]]
        new_high = float(models.predict_rul_from_survival(full_models[subset], row[bundle.full_feature_cols])
                         .pred_ci_high.iloc[0])
        low, old_high = r["confidence_interval_90"]
        rows.append(dict(subset=subset, unit=unit, true_rul=true_rul.loc[unit],
                         low=low, old_high=old_high, new_high=new_high))
test = pd.DataFrame(rows)
test["band"] = pd.cut(test.true_rul, TRUE_BANDS, labels=TRUE_LABELS)
test.to_csv(PROCESSED_DIR / "uncapped_upper_test_check.csv", index=False)

def summarise(g):
    out = {}
    for k, label in [("old_high", "deployed"), ("new_high", "uncapped upper")]:
        cov = g.true_rul.between(g.low, g[k])
        out[(label, "coverage")] = cov.mean()
        out[(label, "width")] = (g[k] - g.low).mean()
        for b in TRUE_LABELS:
            out[(label, b)] = cov[g.band == b].mean()
    return pd.Series(out)

confirmation = test.groupby("subset").apply(summarise, include_groups=False).T
display(confirmation.round(3))
passed = all(confirmation.loc[("uncapped upper", k), s] >= 0.85
             for s in SUBSETS for k in ["coverage", "true <= 25", "true > 125"])
print("Confirmation on the test sets:", "PASSED" if passed else "FAILED")

subset                        FD001    FD004
deployed       coverage       0.970    0.887
               width         99.680  100.049
               true <= 25     1.000    0.940
               true 26-125    0.971    0.947
               true > 125     0.909    0.731
uncapped upper coverage       0.990    0.984
               width        127.045  159.847
               true <= 25     1.000    0.940
               true 26-125    0.986    0.992
               true > 125     1.000    1.000

Confirmation on the test sets: PASSED


## 6. Decision: recorded, not deployed

The uncapped leaf-30 upper bound was selected on training data and confirmed on the test sets: coverage above the cap goes from 91% to 100% (FD001) and from 73% to 100% (FD004). It is still **not deployed**, for four reasons:

1. **The misses it fixes are on the safe side.** An engine that outlives its interval harms nobody. Notebook 15's fix mattered because those misses promised *more* life than the engine had; these promise less.
2. **The upper bound barely drives decisions.** Maintenance is scheduled off how soon an engine could fail. For an engine the model already calls healthy, "80–220 cycles" and "80–300 cycles" lead to the same action: nothing yet.
3. **It overshoots the target.** Coverage would go to about 98–99% against a nominal 90%, swapping a slight under-coverage for a clear over-coverage, with intervals about 27 (FD001) and 60 (FD004) cycles wider on average.
4. **It costs real resources:** a second forest per subset to export, version and keep in step (about 200 MB), more memory on the free live-demo tier, and a longer Docker build.

What would change the decision: a use that needs a trustworthy upper bound, such as fleet planning on how long healthy engines will last, or a requirement for ≥ 90% coverage in every band. The code to build it is all here, through `interval_calibration.out_of_fold_intervals(cap=...)`.

The deployed interval is unchanged: lower bound from the more cautious of RSF and Weibull, upper bound from Weibull, all trained with the 125-cycle cap.